# Run Experiments (Clustering)

This notebook reads `clustering_experiments.yaml`, runs K-Means once for each experiment, and shows a table so you can compare them.

**Before you run it:** run `inspect_and_clean_h2.py` once. It creates `data/processed/h2_clean_all_columns.csv`.

**Run the cells from top to bottom** (one at a time with Shift+Enter, or use "Run All").

## Step 1: Import the tools

In [1]:
import os
import yaml
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Step 2: Go to the main project folder
This notebook lives in the `notebooks` folder, but the YAML file and the data are in the main folder.
This cell moves Python up one folder if needed, so every path works.

In [2]:
# If we are inside the "notebooks" folder, go up one folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working folder:", os.getcwd())

Working folder: /Users/mohamedatef/ML Project/Hotel_Booking_ML-


## Step 3: Read the config file

In [3]:
with open("config/datasets/clustering_experiments.yaml") as file:
    config = yaml.safe_load(file)

seed = config["random_seed"]
number_of_clusters = config["number_of_clusters"]

print("Seed:", seed)
print("Number of clusters:", number_of_clusters)
print("Experiments to run:", list(config["experiments"].keys()))

Seed: 42
Number of clusters: 4
Experiments to run: ['baseline', 'without_waiting_info', 'with_waiting_days', 'with_deposit_and_parking']


## Step 4: Load the data

In [4]:
df = pd.read_csv(config["data_file"])
print("Data loaded:", df.shape)
df.head()

Data loaded: (53042, 34)


,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate,ArrivalDate,ArrivalWeekday,WasWaitlisted
0,0,6,2015,July,27,1,0,2,1,0.0,...,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01,Wednesday,0
1,1,88,2015,July,27,1,0,4,2,0.0,...,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01,Wednesday,0
2,1,65,2015,July,27,1,0,4,1,0.0,...,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01,Wednesday,0
3,1,92,2015,July,27,1,2,4,2,0.0,...,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01,Wednesday,0
4,1,100,2015,July,27,2,0,2,2,0.0,...,0,Transient,76.5,0,1,Canceled,2015-04-02,2015-07-02,Thursday,0


## Step 5: Run every experiment
For each experiment in the YAML file we:
1. build the list of columns,
2. scale the numbers and turn the text into 0/1 columns,
3. run K-Means,
4. measure the result.

This cell takes about a minute.

In [5]:
results = []

for name, settings in config["experiments"].items():
    # An experiment with nothing under its name is read as None, so use an empty dict
    if settings is None:
        settings = {}

    # 5a. Build the list of columns for this experiment
    columns = list(config["base_columns"])
    for column in settings.get("add", []):
        columns.append(column)
    for column in settings.get("remove", []):
        columns.remove(column)

    data = df[columns]

    # 5b. Split the columns into numbers and text
    number_columns = data.select_dtypes(include="number").columns
    text_columns = data.select_dtypes(exclude="number").columns

    # 5c. Scale the numbers, and turn the text into 0/1 columns
    preparation = ColumnTransformer([
        ("numbers", StandardScaler(), number_columns),
        ("text", OneHotEncoder(sparse_output=False), text_columns),
    ])
    X = preparation.fit_transform(data)

    # 5d. Run K-Means
    model = KMeans(n_clusters=number_of_clusters, random_state=seed, n_init=10)
    labels = model.fit_predict(X)

    # 5e. Measure the result
    silhouette = silhouette_score(X, labels, sample_size=10000, random_state=seed)
    davies_bouldin = davies_bouldin_score(X, labels)
    smallest_cluster = pd.Series(labels).value_counts().min()

    results.append({
        "experiment": name,
        "columns": len(columns),
        "silhouette (higher is better)": round(silhouette, 3),
        "davies_bouldin (lower is better)": round(davies_bouldin, 3),
        "smallest_cluster_size": smallest_cluster,
    })
    print("Finished:", name)

Finished: baseline
Finished: without_waiting_info
Finished: with_waiting_days
Finished: with_deposit_and_parking


## Step 6: Compare the results
Read the three numbers **together**. A tiny `smallest_cluster_size` means one group is very small, which can make the scores look better than they really are.

In [6]:
results_table = pd.DataFrame(results)
results_table

,experiment,columns,silhouette (higher is better),davies_bouldin (lower is better),smallest_cluster_size
0,baseline,19,0.129,1.935,710
1,without_waiting_info,18,0.093,2.302,1548
2,with_waiting_days,19,0.094,2.254,1547
3,with_deposit_and_parking,21,0.110,2.123,1549


## Step 7: Save the table

In [7]:
os.makedirs("outputs", exist_ok=True)
results_table.to_csv("outputs/clustering_experiment_results.csv", index=False)
print("Saved to outputs/clustering_experiment_results.csv")

Saved to outputs/clustering_experiment_results.csv


## How to try your own experiment
1. Open `clustering_experiments.yaml`.
2. Add a new block under `experiments:` (keep the same 2-space indentation), for example:

```yaml
  without_special_requests:
    remove: [TotalOfSpecialRequests]
```

3. Save the YAML file.
4. Run this notebook again from Step 3 (the YAML is read again there).

You can also change `number_of_clusters` in the YAML to test a different number of groups.